In [ ]:
import pandas as pd
import numpy as np
import re
#import torch
#import gc
#from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRAIN_PATH = 'DATA_DIR/Train.csv'
VAL_PATH = 'DATA_DIR/Val.csv'
TEST_PATH = 'DATA_DIR/Test.csv'
clean_v0 ="DATA_DIR/optimized_train.csv"

train_df = pd.read_csv(clean_v0)
test_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

# Merge validation into training to maximize available supervision (Charmq GM strategy)
df = pd.concat([train_df, val_df, test_df], ignore_index=True)
#df_eng = clustered


In [ ]:
import pandas as pd
from collections import Counter

# Load raw dataset files
BASE = 'DATA_DIR'
try:
    #train_df = pd.read_csv(f'{BASE}/Train.csv')
    #val_df = pd.read_csv(f'{BASE}/Val.csv')
    
    # Isolate the Amharic Ethiopic subset
    tr_amh = train_df[train_df['subset'] == 'Amh_Eth'].copy()
    vl_amh = val_df[val_df['subset'] == 'Amh_Eth'].copy()
    
    print(f"📊 Amh_Eth Dataset Proportions:")
    print(f"   Train Records: {len(tr_amh)} | Val Records: {len(vl_amh)}")
    
    if len(tr_amh) > 0:
        print("\n📝 Sample Input/Output Structure:")
        print(f"   Input Q:  {tr_amh['input'].iloc[0][:150]}")
        print(f"   Output A: {tr_amh['output'].iloc[0][:150]}")
        
        # Calculate distinct vocabulary intersections
        def get_vocab(series):
            return set(" ".join(series.astype(str)).lower().split())
            
        q_vocab = get_vocab(tr_amh['input'])
        a_vocab = get_vocab(tr_amh['output'])
        val_q_vocab = get_vocab(vl_amh['input'])
        
        print("\n🔍 Vocabulary Coverage Vectors:")
        print(f"   Distinct tokens in Train Questions (Q): {len(q_vocab)}")
        print(f"   Distinct tokens in Train Answers (A):   {len(a_vocab)}")
        print(f"   Lexical Overlap (Q ∩ A):                {len(q_vocab.intersection(a_vocab))}")
        print(f"   Out-Of-Vocabulary Val Qs (Val_Q \ Train_Q): {len(val_q_vocab - q_vocab)}")
        
        # Detect presence of specific Ethiopic punctuation markers
        all_text = " ".join(tr_amh['input'].astype(str))
        punc_markers = ['፡', '።', '፣', '፤', '፥', '፦', '፧', '፨']
        print("\n📌 Ethiopic Punctuation Marker Frequency Counts:")
        for pm in punc_markers:
            print(f"   Marker '{pm}': {all_text.count(pm)}")
            
except Exception as e:
    print(f"❌ Diagnostic Failure: {str(e)}")


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

BASE = 'DATA_DIR'
tr_amh = pd.read_csv(f'{BASE}/Train.csv').query("subset == 'Amh_Eth'").copy()
vl_amh = pd.read_csv(f'{BASE}/Val.csv').query("subset == 'Amh_Eth'").copy()

# Text Standardization
for df in (tr_amh, vl_amh):
    for col in ('input', 'output'):
        df[col] = df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

val_to_train, val_held = train_test_split(vl_amh, test_size=0.10, random_state=42)
pool_amh = pd.concat([tr_amh, val_to_train], ignore_index=True)

def amharic_clean_tokenizer(text):
    t = re.sub(r'[፡።፣፤፥፦፤፧፨?!.,;:\'"\-]+', ' ', str(text).strip())
    return [w for w in t.split() if w]

def compute_f1(pred, ref):
    pt = amharic_clean_tokenizer(pred)
    rt = amharic_clean_tokenizer(ref)
    if not pt or not rt: return 0.0
    pc, rc = Counter(pt), Counter(rt)
    return 2 * sum((pc & rc).values()) / (len(pt) + len(rt))

pool_inputs = pool_amh['input'].tolist()
pool_outputs = pool_amh['output'].tolist()

# Fit vocabulary mapping
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), preprocessor=lambda x: " ".join(amharic_clean_tokenizer(x)))
vec.fit(pool_inputs + pool_outputs)

X_pool = vec.transform(pool_inputs)
X_val = vec.transform(val_held['input'].tolist())

print("🔍 EXAMINING SAMPLE CONTEXT ALIGNMENTS (Ultra-Low Threshold Scan):\n")

# Test real low-bound thresholds to allow matches to pass through
for thresh in [0.05, 0.10, 0.15, 0.20]:
    scores = []
    for idx, row in val_held.reset_index(drop=True).head(3).iterrows():
        sims = cosine_similarity(X_val[idx], X_pool).flatten()
        best_idx = sims.argmax()
        max_sim = sims[best_idx]
        
        passed = max_sim >= thresh
        pred = pool_outputs[best_idx] if passed else "FALLBACK_TRUNCATED_INPUT"
        f1 = compute_f1(pred, row['output'])
        scores.append(f1)
        
        if thresh == 0.10: # Print tracing diagnostics at the critical sweet spot
            print(f"--- Sample {idx} (Sim Max: {max_sim:.4f} | Pass T=0.10: {passed}) ---")
            print(f"Q: {row['input'][:90]}")
            print(f"P: {pred[:90]}")
            print(f"G: {row['output'][:90]}")
            print(f"F1 Alignment Score: {f1:.4f}\n")
            
    print(f"📊 Mean Blended Score at Threshold {thresh:.2f} -> {np.mean(scores):.4f}")


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE = 'DATA_DIR'
tr_amh = pd.read_csv(f'{BASE}/Train.csv').query("subset == 'Amh_Eth'").copy()
vl_amh = pd.read_csv(f'{BASE}/Val.csv').query("subset == 'Amh_Eth'").copy()

def amharic_tokenizer(text):
    t = re.sub(r'[፡።፣፤፥፦፤፧፨?!.,;:\'"\-]+', ' ', str(text).strip().lower())
    return [w for w in t.split() if len(w) > 1]

pool_inputs = tr_amh['input'].tolist()
pool_outputs = tr_amh['output'].tolist()
val_df = vl_amh.reset_index(drop=True)

vec = CountVectorizer(tokenizer=amharic_tokenizer, binary=True)
X_pool = vec.fit_transform(pool_inputs)
X_val = vec.transform(val_df['input'].tolist())

print("🕵️ Analyzing Low-Confidence Amharic Alignment Mismatches:\n")

sample_indices = [5, 12, 25]  # Targeted diagnostic index slice
for idx in sample_indices:
    if idx >= len(val_df): break
    row = val_df.iloc[idx]
    
    sims = cosine_similarity(X_val[idx], X_pool).flatten()
    best_idx = sims.argmax()
    max_sim = sims[best_idx]
    
    q_tokens = amharic_tokenizer(row['input'])
    p_tokens = amharic_tokenizer(pool_outputs[best_idx])
    g_tokens = amharic_tokenizer(row['output'])
    
    # Calculate exact intersecting features
    intersect_pq = set(q_tokens).intersection(set(p_tokens))
    intersect_gq = set(q_tokens).intersection(set(g_tokens))
    
    print(f"=== Diagnostic Instance {idx} [Max Cosine Sim: {max_sim:.4f}] ===")
    print(f"❓ Q (Input):  {row['input']}")
    print(f"🎯 P (Pred):   {pool_outputs[best_idx]}")
    print(f"🟢 G (Truth):  {row['output']}")
    print(f"⚔️ Shared tokens with Prediction (Q ∩ P): {list(intersect_pq)}")
    print(f"⚔️ Shared tokens with Ground Truth (Q ∩ G): {list(intersect_gq)}")
    print(f"📏 Length Profile -> Q: {len(q_tokens)} | P: {len(p_tokens)} | G: {len(g_tokens)}\n")


In [ ]:
import os, re, warnings
print("📦 Building accelerated Hybrid Fusion Engine...")
os.system("pip install sentence-transformers rouge_score --quiet")

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer as rs_module

warnings.filterwarnings('ignore')
_ROUGE = rs_module.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# ── 1. LOAD PARTITIONS AND DATASETS ─────────────────────────────────────────
BASE = 'DATA_DIR'
tr_amh = pd.read_csv(f'{BASE}/Train.csv').query("subset == 'Amh_Eth'").copy()
vl_amh = pd.read_csv(f'{BASE}/Val.csv').query("subset == 'Amh_Eth'").copy()

# Max optimization slice to assure instant CPU calculation execution profile
pool_inputs = tr_amh['input'].tolist()
pool_outputs = tr_amh['output'].tolist()
val_inputs = vl_amh['input'].tolist()

def amharic_clean_tokens(text):
    t = re.sub(r'[፡።፣፤፥፦፤፧፨?!.,;:\'"\-]+', ' ', str(text).strip().lower())
    return [w for w in t.split() if len(w) > 1]

# ── 2. SPARSE LEXICAL MATCHING LAYER ────────────────────────────────────────
vec_sparse = TfidfVectorizer(tokenizer=amharic_clean_tokens, sublinear_tf=True, ngram_range=(1, 2))
X_pool_sparse = vec_sparse.fit_transform(pool_inputs)
X_val_sparse = vec_sparse.transform(val_inputs)
sims_sparse = cosine_similarity(X_val_sparse, X_pool_sparse)

# ── 3. DENSE SEMANTIC EMBEDDING LAYER (CPU) ─────────────────────────────────
model = SentenceTransformer('rasyosef/bert-amharic-text-embedding-medium', device='cpu')
X_pool_dense = model.encode(pool_inputs, convert_to_tensor=True, show_progress_bar=False)
X_val_dense = model.encode(val_inputs, convert_to_tensor=True, show_progress_bar=False)
sims_dense = util.cos_sim(X_val_dense, X_pool_dense).cpu().numpy()

# ── 4. HYBRID RECIPROCAL RANK FUSION MATRICES ──────────────────────────────
# Weights are balanced to lock exact keywords while utilizing the semantic space context
hybrid_sims = (0.45 * sims_sparse) + (0.55 * sims_dense)

best_indices = hybrid_sims.argmax(axis=1)
max_sims = hybrid_sims.max(axis=1)

# ── 5. EVALUATE METRICS AND TRACE ───────────────────────────────────────────
scores_r1, scores_rl = [], []

for idx, (best_idx, max_sim) in enumerate(zip(best_indices, max_sims)):
    row = vl_amh.iloc[idx]
    
    # Tuned fallback threshold to route safe passes
    pred = pool_outputs[best_idx] if max_sim >= 0.32 else row['input']
    
    pt = amharic_clean_tokens(pred)
    rt = amharic_clean_tokens(row['output'])
    
    if pt and rt:
        r_scores = _ROUGE.score(" ".join(rt), " ".join(pt))
        scores_r1.append(r_scores['rouge1'].fmeasure)
        scores_rl.append(r_scores['rougeL'].fmeasure)
    else:
        scores_r1.append(0.0)
        scores_rl.append(0.0)

blended = 0.5 * np.mean(scores_r1) + 0.5 * np.mean(scores_rl)
print(f"\n📊 Hybrid Lexical-Semantic Amharic Score: {blended:.4f}")


In [ ]:
from rouge_score import rouge_scorer
from collections import Counter

def _lcs_length(a, b):
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    prev = [0] * (n + 1)
    for i in range(m):
        curr = [0] * (n + 1)
        for j in range(n):
            if a[i] == b[j]:
                curr[j+1] = prev[j] + 1
            else:
                curr[j+1] = max(curr[j], prev[j+1])
        prev = curr
    return prev[n]
def score_ethiopic(pred: str, ref: str) -> dict:
    def tok(text):
        t = re.sub(r'[፡።፣፤፥፦\u1361-\u1368\s]+', ' ', str(text)).strip().lower()
        return [w for w in t.split() if w]

    pt = tok(pred)
    rt = tok(ref)

    if not pt or not rt:
        return {'rouge1': 0.0, 'rougeL': 0.0, 'blended': 0.0}

    pc, rc = Counter(pt), Counter(rt)
    common = sum((pc & rc).values())
    p1 = common / len(pt)
    r1 = common / len(rt)
    f1 = 2*p1*r1/(p1+r1) if (p1+r1) > 0 else 0.0

    lcs = _lcs_length(pt, rt)
    pl = lcs / len(pt)
    rl = lcs / len(rt)
    fL = 2*pl*rl/(pl+rl) if (pl+rl) > 0 else 0.0

    return {'rouge1': round(f1,4), 'rougeL': round(fL,4),
            'blended': round(0.5*f1+0.5*fL, 4)}    


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# Local path configs
TRAIN_PATH = 'DATA_DIR/Train.csv'
VAL_PATH   = 'DATA_DIR/Val.csv'

# Standalone template configurations to prevent global memory dependency failures
ENG_GHA_TEMPLATES = {
    'cse_education': "Comprehensive sexuality education (CSE) plays a critical role in promoting adolescents' understanding of gender equality, consent, healthy relationships, and reproductive rights...",
    'default': "Adolescents have the right to access comprehensive sexual and reproductive health education and services..."
}

def quick_gha_template(question):
    q = str(question).lower()
    if any(w in q for w in ['cse','curriculum','gender','stereotype','equality','norm']):
        return ENG_GHA_TEMPLATES['cse_education']
    return ENG_GHA_TEMPLATES['default']

# Executing local evaluation tracking
try:
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    
    sub_train = train_df[train_df['subset'] == 'Eng_Gha'].copy()
    sub_val = val_df[val_df['subset'] == 'Eng_Gha'].copy()
    
    # Analyze text profiles directly
    train_lens = sub_train['output'].astype(str).apply(lambda x: len(x.split()))
    val_lens = sub_val['output'].astype(str).apply(lambda x: len(x.split()))
    
    print("=== Eng_Gha Structural Diagnostics ===")
    print(f"Train Output - Average Word Length: {train_lens.mean():.2f}")
    print(f"Val Output   - Average Word Length: {val_lens.mean():.2f}")
    print(f"Train Output - Min: {train_lens.min()} words | Max: {train_lens.max()} words")
    print(f"Val Output   - Min: {val_lens.min()} words | Max: {val_lens.max()} words")
    
    # Check for direct overlap matches between splits
    train_inputs = set(sub_train['input'].astype(str).str.lower().str.strip())
    val_inputs = set(sub_val['input'].astype(str).str.lower().str.strip())
    exact_matches = val_inputs.intersection(train_inputs)
    
    print(f"\n=== Split Target Gaps ===")
    print(f"Exact Question String Overlap between Splits: {len(exact_matches)} / {len(sub_val)}")
    
    # Print sample text combinations to trace ROUGE penalty anomalies
    print("\n=== Word Profile Sample Traces ===")
    for idx, row in sub_val.head(2).iterrows():
        print(f"\nQ: {row['input']}")
        print(f"Ground Truth Target Answer: {row['output']}")
        print(f"Fallback Paragraph Target:  {quick_gha_template(row['input'])}")

except Exception as e:
    print(f"Data read error: {e}. Confirm paths match Kaggle workspace input structures.")


In [ ]:
import pandas as pd
import numpy as np

TRAIN_PATH = 'DATA_DIR/Train.csv'
VAL_PATH   = 'DATA_DIR/Val.csv'

try:
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    
    sub_train = train_df[train_df['subset'] == 'Amh_Eth']
    sub_val = val_df[val_df['subset'] == 'Amh_Eth']
    
    print(f"=== Random 10 Samples from Eng_Gha Validation Set ===\n")
    
    # Draw a representative seed-locked random sample
    samples = sub_val.sample(n=10, random_state=42)
    
    for idx, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"[{idx}] QUESTION:")
        print(f"    {row['input']}")
        print(f"    GROUND TRUTH ANSWER:")
        print(f"    {row['output']}\n" + "-"*50)
        
except Exception as e:
    print(f"Execution Error: {e}")


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

TRAIN_PATH = 'DATA_DIR/Train.csv'
VAL_PATH = 'DATA_DIR/Val.csv'
TEST_PATH = 'DATA_DIR/Test.csv'
OUTPUT_TRAIN_PATH = 'optimized_train.csv'
OUTPUT_VAL_PATH = 'optimized_val.csv'

# Dynamic character percentiles per subset for training-only filtering
dialect_percentiles = {
    'Amh_Eth': {'min_q': 0.10, 'max_q': 0.75}, 
    'Eng_Eth': {'min_q': 0.10, 'max_q': 0.75}, 
    'Eng_Gha': {'min_q': 0.05, 'max_q': 0.90}, 
    'Eng_Ken': {'min_q': 0.05, 'max_q': 0.90},
    'Eng_Uga': {'min_q': 0.05, 'max_q': 0.90},
    'Lug_Uga': {'min_q': 0.05, 'max_q': 0.90},
    'Swa_Ken': {'min_q': 0.05, 'max_q': 0.90},
    'Aka_Gha': {'min_q': 0.03, 'max_q': 0.95}, 
}

print("Loading datasets...")
train_raw = pd.read_csv(TRAIN_PATH)
val_raw = pd.read_csv(VAL_PATH)
test_raw = pd.read_csv(TEST_PATH)

# ── Phase 1: Stratified 80/20 Val Split Allocation ──
print("Splitting validation set 80/20...")
val_to_train_list = []
val_to_eval_list = []

for subset, group in val_raw.groupby('subset'):
    train_part = group.sample(frac=0.8, random_state=42)
    eval_part = group.drop(train_part.index)
    val_to_train_list.append(train_part)
    val_to_eval_list.append(eval_part)

allocated_val_train = pd.concat(val_to_train_list, ignore_index=True)
allocated_val_eval = pd.concat(val_to_eval_list, ignore_index=True)

train_df = pd.concat([train_raw, allocated_val_train], ignore_index=True)
eval_df = allocated_val_eval.copy()

# ── Phase 2: Basic Type Normalisation & Spacing ──
def normalize_basic(df):
    for col in ["input", "output"]:
        df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    df = df.dropna(subset=["input", "output", "subset", "ID"])
    df = df[df["input"].str.len() > 0]
    df = df[df["output"].str.len() > 0]
    return df

train_df = normalize_basic(train_df)
eval_df = normalize_basic(eval_df)

# ── Phase 3: Strict Test Set Leakage Prevention ──
print("Preventing validation and test leakages...")
test_inputs = set(test_raw['input'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.lower())

train_df = train_df[~train_df['input'].str.lower().isin(test_inputs)]
eval_df = eval_df[~eval_df['input'].str.lower().isin(test_inputs)]

# ── Phase 4: Deduplication (Case-insensitive) ──
def deduplicate_normalized(df):
    df["input_norm"] = df["input"].str.lower().str.strip()
    before = len(df)
    df = df.drop_duplicates(subset=["subset", "input_norm"], keep="first")
    print(f"Removed {before - len(df)} duplicate inputs.")
    return df

train_df = deduplicate_normalized(train_df)
eval_df = deduplicate_normalized(eval_df)

# ── Phase 5: Tautology, Noise, and Semantic TTR Filters ──
def overlap_ratio(a, b):
    a_words = set(a.lower().split())
    b_words = set(b.lower().split())
    if not a_words or not b_words:
        return 0.0
    return len(a_words & b_words) / len(a_words | b_words)

NOISE_PATTERNS = [
    r"^\s*n/?a\s*$",
    r"^(none|null|undefined|nan)\s*$",
    r"https?://\S+",
    r"^[\W\d\s]{1,10}$",
    r"\berror\b",
    r"<[^>]+>",
    r"\[.*?\].*?\[.*?\]",
    r"^\s*\?\s*$",
]

def is_noise(text):
    t = str(text).strip().lower()
    return any(re.search(p, t) for p in NOISE_PATTERNS)

def type_token_ratio(text):
    words = str(text).lower().split()
    if len(words) < 4:
        return 1.0
    return len(set(words)) / len(words)

def apply_semantic_filters(df, label="data"):
    # Tautology Filter
    df["tautology_score"] = df.apply(lambda r: overlap_ratio(r["input"], r["output"]), axis=1)
    before = len(df)
    df = df[df["tautology_score"] < 0.85]
    print(f"[{label}] After tautology filter: {len(df)} (removed {before - len(df)})")
    
    # Noise pattern filter
    before = len(df)
    df = df[~df["output"].apply(is_noise)]
    df = df[~df["input"].apply(is_noise)]
    print(f"[{label}] After noise filter: {len(df)} (removed {before - len(df)})")
    
    # Type-Token Ratio
    df["ttr"] = df["output"].apply(type_token_ratio)
    before = len(df)
    df = df[df["ttr"] > 0.25]
    print(f"[{label}] After TTR filter: {len(df)} (removed {before - len(df)})")
    
    return df

print("Applying semantic cleansers...")
train_df = apply_semantic_filters(train_df, "Train")
eval_df = apply_semantic_filters(eval_df, "Val Eval")

# ── Phase 6: Rare Character Removal ──
print("Analyzing training corpus for character-level noise...")
all_train_text = " ".join(train_df['input'].tolist() + train_df['output'].tolist())
char_counts = Counter(all_train_text)
rare_chars = {char for char, count in char_counts.items() if count < 5}

def has_rare_char(text):
    return any(c in rare_chars for c in text)

train_df = train_df[~(train_df['input'].apply(has_rare_char) | train_df['output'].apply(has_rare_char))]
eval_df = eval_df[~(eval_df['input'].apply(has_rare_char) | eval_df['output'].apply(has_rare_char))]

# ── Phase 7: Flag Repeated Duplicate Answers ──
train_df["output_norm"] = train_df["output"].str.lower().str.strip()
output_counts = train_df["output_norm"].value_counts()
suspicious_outputs = set(output_counts[output_counts > 15].index)
train_df["is_duplicate_answer"] = train_df["output_norm"].isin(suspicious_outputs)
print(f"Flagged {train_df['is_duplicate_answer'].sum()} high-frequency duplicate answers.")

# ── Phase 8: Dynamic Character-Length Percentile Outliers ──
filtered_train_dfs = []
for subset, qs in dialect_percentiles.items():
    sub_df = train_df[train_df['subset'] == subset]
    in_lens, out_lens = sub_df['input'].str.len(), sub_df['output'].str.len()
    
    clean_sub = sub_df[
        (in_lens >= in_lens.quantile(qs['min_q'])) & (in_lens <= in_lens.quantile(qs['max_q'])) &
        (out_lens >= out_lens.quantile(qs['min_q'])) & (out_lens <= out_lens.quantile(qs['max_q']))
    ]
    filtered_train_dfs.append(clean_sub)

train_df = pd.concat(filtered_train_dfs, ignore_index=True)

# ── Phase 9: Task Prefixing ──
train_df['input'] = train_df['subset'] + ": " + train_df['input']
eval_df['input'] = eval_df['subset'] + ": " + eval_df['input']

# ── Phase 10: Clean Helper Columns and Save ──
cols_to_drop = ["input_norm", "output_norm", "tautology_score", "ttr"]
for c in cols_to_drop:
    if c in train_df.columns:
        train_df = train_df.drop(columns=[c])
    if c in eval_df.columns:
        eval_df = eval_df.drop(columns=[c])

train_df.to_csv(OUTPUT_TRAIN_PATH, index=False)
eval_df.to_csv(OUTPUT_VAL_PATH, index=False)

print("\n=== PIPELINE SUCCESS ===")
print(f"Optimized Train: {len(train_df)} rows")
print(f"Optimized Val: {len(eval_df)} rows")


In [ ]:
import pandas as pd
import re

# Load raw datasets
train = pd.read_csv('DATA_DIR/Train.csv')
val = pd.read_csv('DATA_DIR/Val.csv')
df = pd.concat([train, val], ignore_index=True)

# Preprocessing: Replace URLs with a clean token rather than dropping the whole row
def clean_urls(text):
    return re.sub(r'https?://\S+', '[URL]', str(text))

df['input_clean'] = df['input'].apply(clean_urls)
df['output_clean'] = df['output'].apply(clean_urls)

# Refined strict noise patterns
NOISE_PATTERNS = {
    "Bare N/A": r"^\s*n/?a\s*$",
    "Null/Nan Words": r"^(none|null|undefined|nan)\s*$",
    "Punctuation/Number Garbage": r"^[\W\d\s]{1,10}$",
    "HTML Tags": r"<[^>]+>",
    "Bare Question Marks": r"^\s*\?\s*$"
}

print("=== REFINED NOISE PATTERN DIAGNOSTIC REPORT ===")
for name, pattern in NOISE_PATTERNS.items():
    match_in = df['input_clean'].astype(str).str.contains(pattern, flags=re.IGNORECASE, regex=True, na=False)
    match_out = df['output_clean'].astype(str).str.contains(pattern, flags=re.IGNORECASE, regex=True, na=False)
    
    matched_df = df[match_in | match_out]
    total_found = len(matched_df)
    
    print(f"\n🔍 Pattern: {name} | Matches Found: {total_found}")
    if total_found > 0:
        samples = matched_df.sample(min(3, total_found), random_state=42)
        for _, row in samples.iterrows():
            is_in = bool(re.search(pattern, str(row['input_clean']), re.IGNORECASE))
            matched_text = row['input_clean'] if is_in else row['output_clean']
            print(f"    - [{row['subset']}]: {repr(str(matched_text)[:120])}...")


In [ ]:
import pandas as pd
import numpy as np

# 1. Load data
train_df = pd.read_csv('optimized_train.csv')
#audit = pd.read_csv('eng_14_cluster_audit - eng_14_cluster_audit.csv')
id_map = dict(zip(audit['ID'], audit['merged_group']))

category_map = {
    'G01_ADOLESCENT_SRH': 'ADOLESCENT SRH',
    'G02_HIV_GENERAL': 'HIV GENERAL',
    'G03_HIV_TRANSMISSION': 'HIV TRANSMISSION',
    'G04_HERPES_HSV': 'HERPES',
    'G05_SYPHILIS': 'SYPHILIS',
    'G06_CHLAMYDA': 'CHLAMYDIA',
    'G06_CHLAMYDIA': 'CHLAMYDIA',
    'G07_GONORRHEA': 'GONORRHEA',
    'G08_HEPATITIS_HBV': 'HEPATITIS',
    'G09_TRICHOMONIASIS': 'TRICHOMONIASIS',
    'G10_HPV': 'HPV',
    'G11_PEP_PREP': 'PEP PREP',
    'G12_GENERAL_STIS': 'GENERAL STIS',
    'G13_CULTURAL_MYTHS': 'CULTURAL_MYTHS',
    'G14_FALLBACK': 'FALLBACK'
}

# Define Clinical Keyword Dictionaries for English + Rest of Dialects (Amh, Swa, Lug, Aka)
CLINICAL_DICTS = {
    'ADOLESCENT SRH': [
        'puberty', 'adolescent', 'teen', 'peer pressure', 'hormone', 
        'financial empowerment', 'pregnancy', 'contraceptive', 'condom', 'srh',
        'mimba', 'kondomu', 'lubuto', 'kapiira', 'nyinsɛn', 'kɔndɔm', 'እርግዝና', 'ኮንዶም'
    ],
    'HIV GENERAL': [
        'hiv', 'aids', 'arv', 'art', 'retroviral', 'viral load', 'cd4',
        'ukimwi', 'vvu', 'siriimu', 'akawuka', 'ኤችአይቪ', 'ኤድስ'
    ],
    'HIV TRANSMISSION': [
        'transmit', 'transmission', 'spread hiv', 'catch hiv', 'mosquito bite', 'saliva',
        'kutambaza', 'kwata'
    ],
    'HERPES': [
        'herpes', 'hsv', 'cold sore', 'blister', 'acyclovir', 'ሄርፒስ'
    ],
    'SYPHILIS': [
        'syphilis', 'chancre', 'vdrl', 'rpr', 'kaswende', 'kabotongo', 'ቂጥኝ'
    ],
    'CHLAMYDIA': [
        'chlamydia', 'ክላሚዲያ'
    ],
    'GONORRHEA': [
        'gonorrhea', 'clap', 'gonorrhoea', 'kisonono', 'nzege', 'babaso', 'ጨብጥ'
    ],
    'HEPATITIS': [
        'hepatitis', 'hbv', 'hepb', 'liver', 'homa ya ini', 'ሄፓታይተስ'
    ],
    'TRICHOMONIASIS': [
        'trichomoniasis', 'trich', 'ትሪኮሞኒያሲስ'
    ],
    'HPV': [
        'hpv', 'genital wart', 'cervical cancer'
    ],
    'PEP PREP': [
        'prep', 'pep', 'truvada', 'post-exposure'
    ],
    'GENERAL STIS': [
        'sti', 'std', 'venereal', 'infection', 'ndwara', 'yareza'
    ],
    'CULTURAL_MYTHS': [
        'cure', 'heal hiv', 'prayer', 'myth', 'witchcraft', 'lemon', 'garlic',
        'dawa ya kienyeji', 'mti', 'kitunguu swumu', 'ሎሚ', 'ነጭ ሽንኩርት'
    ]
}

# Heuristic classifier matching function
def dictionary_classify(text):
    text_lower = str(text).lower()
    for category, keywords in CLINICAL_DICTS.items():
        if any(kw in text_lower for kw in keywords):
            return category
    return 'FALLBACK'

train_df['target_group'] = train_df['ID'].map(id_map).map(category_map)

# Predict on unlabeled rows with pure Dictionary Routing (All dialects included)
unlabeled = train_df[train_df['target_group'].isna()].copy()
if len(unlabeled) == 0:
    unlabeled = train_df.copy()

predictions = []
confidences = []

for idx, row in unlabeled.iterrows():
    text_val = str(row['input'])
    dict_pred = dictionary_classify(text_val)
    predictions.append(dict_pred)
    confidences.append(1.00 if dict_pred != 'General Health' else 0.00)

unlabeled['pred'] = predictions
unlabeled['conf'] = confidences

# 3. Print 10 random predictions across all dialects
sampled = unlabeled.sample(10, random_state=42)
print("=== Dictionary-Only Predictions (All Dialects) ===")
for _, r in sampled.iterrows():
    print(f"[{r['subset']}] Pred: {r['pred']} (Conf: {r['conf']:.2f})\nQ: {r['input'][:110]}...\n")


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


# ── 2. DATA PURGE & CONSOLIDATION ────────────────────────────────────────────
print("📖 Loading CSV volumes into memory...")
df_train_raw = pd.read_csv(TRAIN_PATH)
df_val_raw = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)

# Standardize verification strings to eliminate baseline text duplicates
df_test["input_clean"] = df_test["input"].astype(str).str.lower().str.strip()
test_inputs_set = set(df_test["input_clean"].tolist())

print("🗂️ Squeezing data splits into a unified training collection...")
df_all = pd.concat([df_train_raw, df_val_raw], ignore_index=True)
df_all["input_clean"] = df_all["input"].astype(str).str.lower().str.strip()

# AUDIT REMARK: Cross-dataset duplicate leak purge 
initial_count = len(df_all)
df_train_clean = df_all[~df_all["input_clean"].isin(test_inputs_set)].copy()
df_train_clean = df_train_clean.drop_duplicates(subset=["input_clean"], keep="first").reset_index(drop=True)

print(f"✂️ Removed {initial_count - len(df_train_clean)} overlapping leak rows from reference data.")
print(f"📊 Final training reference database capacity: {len(df_train_clean)} records.")

# ── 3. VECTORIZED MATRIX FIT ──────────────────────────────────────────────────
print("⚡ Computing Vectorized Char-TF-IDF spaces across training documents...")
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2)

train_inputs = df_train_clean['input'].astype(str).tolist()
X_train = vectorizer.fit_transform(train_inputs)
X_test  = vectorizer.transform(df_test['input'].astype(str))

print("🚀 Calculating Global Cross-Similarity profiles via dot-products...")
similarity_matrix = X_test.dot(X_train.T)

# Fast axis argmax isolates matching indices and scores instantly
best_match_indices = np.array(similarity_matrix.argmax(axis=1)).flatten()
best_scores = np.array(similarity_matrix.max(axis=1).toarray()).flatten()

df_test['matched_train_idx'] = best_match_indices
df_test['score'] = best_scores

# ── 4. AUDIT REMARK: AUDITED DYNAMIC THRESHOLDS ──────────────────────────────
DYNAMIC_THRESHOLDS = {
    "Eng_Eth": 0.80, "Eng_Gha": 0.80, "Eng_Ken": 0.80, "Eng_Uga": 0.80,
    "Lug_Uga": 0.85, "Swa_Ken": 0.85,
    "Amh_Eth": 0.92, "Aka_Gha": 0.92
}

def evaluate_row_status(row):
    subset = row.get('subset', 'Unknown')
    limit = DYNAMIC_THRESHOLDS.get(subset, 0.92)
    if row['score'] >= limit:
        return 'CACHE HIT'
    elif 0.40 <= row['score'] < limit:
        return 'FEW-SHOT INJECTION'
    return 'LLM FALLBACK'

df_test['status'] = df_test.apply(evaluate_row_status, axis=1)

# ── 5. OUTPUT METRIC LOG DIAGNOSTICS ──────────────────────────────────────────
total_count = len(df_test)
hit_count = (df_test['status'] == 'CACHE HIT').sum()
few_shot_count = (df_test['status'] == 'FEW-SHOT INJECTION').sum()
fallback_count = (df_test['status'] == 'LLM FALLBACK').sum()

print("\n" + "="*60)
print("📈 LAYER 1 CASCADING STATISTICS PROFILE")
print("="*60)
print(f"🎯 Total Cache Hits     : {hit_count}/{total_count} ({hit_count/total_count*100:.2f}%) ──► Safe Exact Returns")
print(f"💡 Total Few-Shot Contexts: {few_shot_count}/{total_count} ({few_shot_count/total_count*100:.2f}%) ──► Injected Examples")
print(f"⚠️ Total Pure Fallbacks   : {fallback_count}/{total_count} ({fallback_count/total_count*100:.2f}%) ──► Direct Generation")
print("="*60)

# ── 6. DYNAMIC MULTI-DIALECT SAMPLER (5 SAMPLES EACH) ────────────────────────
print("\n🔍 Extracting Dialect Profiles...")
dialects = df_test['subset'].unique() if 'subset' in df_test.columns else ['All_Dialects']

for dialect in dialects:
    print("\n" + "="*85)
    print(f"🌍 DIALECT MARKET METRICS: {dialect}")
    print("="*85)
    
    dialect_df = df_test[df_test['subset'] == dialect] if 'subset' in df_test.columns else df_test
    
    # Stratify samples across priorities to display complete behavioral spread
    sample_hits = dialect_df[dialect_df['status'] == 'CACHE HIT'].head(2)
    sample_fshots = dialect_df[dialect_df['status'] == 'FEW-SHOT INJECTION'].head(2)
    sample_miss = dialect_df[dialect_df['status'] == 'LLM FALLBACK'].head(5 - (len(sample_hits) + len(sample_fshots)))
    
    display_samples = pd.concat([sample_hits, sample_fshots, sample_miss]).head(5)
    
    if len(display_samples) == 0:
        print("   No samples found for this subset classification.")
        continue
        
    for i, (_, row) in enumerate(display_samples.iterrows()):
        train_row = df_train_clean.iloc[int(row['matched_train_idx'])]
        
        print(f"📍 Sample {i+1} | Score: {row['score']:.4f} | Status: [{row['status']}]")
        print(f"   Test Inp : {str(row['input']).strip()}")
        print(f"   Train Mat: {str(train_row['input']).strip()}")
        
        # Display cached ground truth outputs for matches or few-shot references
        if 'output' in train_row and pd.notna(train_row['output']):
            ans_preview = str(train_row['output']).strip()
            if row['status'] == 'CACHE HIT':
                print(f"   Cached   : {ans_preview[:120]}..." if len(ans_preview) > 120 else f"   Cached   : {ans_preview}")
            elif row['status'] == 'FEW-SHOT INJECTION':
                print(f"   Context  : {ans_preview[:120]}..." if len(ans_preview) > 120 else f"   Context  : {ans_preview}")
        else:
            print("   Output   : [No training output found in verified reference record]")
            
        print("-" * 75)

# Save intermediate mapping file to disk for diagnostic debugging
df_test.to_csv('/kaggle/working/layer1_retrieval_map.csv', index=False)
print("\nProcessing complete. Retrieval blueprint saved to /kaggle/working/layer1_retrieval_map.csv")
